# FlatX runs with FaIR v2.1.3

where X = 2.5, 5, 10, 20, 40

In [ ]:
import os
import fair
from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties
import numpy as np
import matplotlib.pyplot as pl
import pandas as pd
import pooch
import seaborn as sns
from tqdm.auto import tqdm
import xarray as xr

In [ ]:
fair.__version__

In [ ]:
scenarios = ['esm-flat40', 'esm-flat40_zec', 'esm-flat20', 'esm-flat20_zec', 'esm-flat10', 'esm-flat10_zec', 'esm-flat5', 'esm-flat5_zec', 'esm-flat2.5', 'esm-flat2.5_zec']

In [ ]:
calibration = '1.4.0'
cal_df = {}

In [ ]:
cal_df['1.4.0'] = pd.read_csv('../data/calibration/v1.4.0/calibrated_constrained_parameters.csv', index_col=0)

In [ ]:
species = ['CO2', 'CH4', 'N2O']
properties = {
    "CO2": {
        'type': 'co2',
        'input_mode': 'emissions',
        'greenhouse_gas': True,
        'aerosol_chemistry_from_emissions': False,
        'aerosol_chemistry_from_concentration': False
    },
    "CH4": {
        'type': 'ch4',
        'input_mode': 'emissions',
        'greenhouse_gas': True,
        'aerosol_chemistry_from_emissions': False,
        'aerosol_chemistry_from_concentration': False
    },
    "N2O": {
        'type': 'n2o',
        'input_mode': 'emissions',
        'greenhouse_gas': True,
        'aerosol_chemistry_from_emissions': False,
        'aerosol_chemistry_from_concentration': False
    }
}

In [ ]:
emissions_rate = np.array([40, 20, 10, 5, 2.5], dtype=int)
# ramp_up_length = (1000/emissions_rate).astype(int)
# experiment_length = ramp_up_length + 200
# experiment_length

In [ ]:
f = {}
for experiment in emissions_rate:
    f[experiment] = FAIR()
    ramp_up_length = int(1000/experiment)
    experiment_length = ramp_up_length + 200
    f[experiment].define_time(0, experiment_length, 1)
    f[experiment].define_scenarios([f'esm-flat{experiment}', f'esm-flat{experiment}_zec'])
    f[experiment].define_configs(list(cal_df[calibration].index))
    
    # declare species and properties
    f[experiment].define_species(species, properties)
    
    f[experiment].allocate()
    
    # fill emissions: zero for non-CO2
    f[experiment].emissions.loc[dict(specie="CH4")] = 0
    f[experiment].emissions.loc[dict(specie="N2O")] = 0
    
    # constant pre-industrial concentration for non-CO2 GHGs
    f[experiment].concentration.loc[dict(specie='CH4')] = 808.2490285
    f[experiment].concentration.loc[dict(specie='N2O')] = 273.021047
    
    # fill emissions of CO2 for each scenario
    f[experiment].emissions.loc[dict(specie="CO2", scenario=f'esm-flat{experiment}')] = experiment * 44.009 / 12.011
    f[experiment].emissions.loc[dict(specie="CO2", scenario=f'esm-flat{experiment}_zec', timepoints=np.arange(0.5, ramp_up_length))] = experiment * 44.009 / 12.011
    f[experiment].emissions.loc[dict(specie="CO2", scenario=f'esm-flat{experiment}_zec', timepoints=np.arange(ramp_up_length+0.5, experiment_length))] = 0
    # f[cal].emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr", timepoints=np.arange(0.5, 25))] = 40 * 44.009 / 12.011
    # f[cal].emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr", timepoints=np.arange(25.5, 50))] = np.linspace(38.4, -38.4, 25)[:, None] * 44.009 / 12.011
    # f[cal].emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr", timepoints=np.arange(50.5, 75))] = -40 * 44.009 / 12.011
    # f[cal].emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr", timepoints=np.arange(75.5, 170))] = 0
    
    # Get default species configs
    f[experiment].fill_species_configs()

    # Climate response
    fill(f[experiment].climate_configs['ocean_heat_capacity'], cal_df[calibration].loc[:,'clim_c1':'clim_c3'])
    fill(f[experiment].climate_configs['ocean_heat_transfer'], cal_df[calibration].loc[:,'clim_kappa1':'clim_kappa3'])
    fill(f[experiment].climate_configs['deep_ocean_efficacy'], cal_df[calibration].loc[:,'clim_epsilon'])
    fill(f[experiment].climate_configs['gamma_autocorrelation'], cal_df[calibration].loc[:,'clim_gamma'])
    fill(f[experiment].climate_configs['stochastic_run'], False)

    # carbon cycle
    fill(f[experiment].species_configs['iirf_0'], cal_df[calibration].loc[:, 'cc_r0'].values.squeeze(), specie='CO2')
    fill(f[experiment].species_configs['iirf_airborne'], cal_df[calibration].loc[:, 'cc_rA'].values.squeeze(), specie='CO2')
    fill(f[experiment].species_configs['iirf_uptake'], cal_df[calibration].loc[:, 'cc_rU'].values.squeeze(), specie='CO2')
    fill(f[experiment].species_configs['iirf_temperature'], cal_df[calibration].loc[:, 'cc_rT'].values.squeeze(), specie='CO2')

    # Scale CO2 forcing based on its 4xCO2 calibration
    fill(f[experiment].species_configs["forcing_scale"], cal_df[calibration]["fscale_CO2"].values.squeeze(), specie='CO2')

    # initial condition of CO2 concentration (but not baseline for forcing calculations)
    fill(f[experiment].species_configs['baseline_concentration'], 284.3169988, specie='CO2')
    fill(f[experiment].species_configs['baseline_concentration'], 808.2490285, specie='CH4')
    fill(f[experiment].species_configs['baseline_concentration'], 273.021047, specie='N2O')
    
    # set initial conditions
    initialise(f[experiment].concentration, f[experiment].species_configs['baseline_concentration'])
    initialise(f[experiment].forcing, 0)
    initialise(f[experiment].temperature, 0)
    initialise(f[experiment].airborne_emissions, 0)
    initialise(f[experiment].cumulative_emissions, 0)
    
    f[experiment].run()

In [ ]:
for experiment in emissions_rate:
    fig, ax = pl.subplots(2, 2)
    ax[0,0].plot(f[experiment].timepoints, f[experiment].emissions.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}")], color='k', alpha=0.1);
    ax[0,0].plot(f[experiment].timepoints, f[experiment].emissions.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}_zec")], color='b', alpha=0.1);
    # ax[0,0].plot(f[cal].timepoints, f[cal].emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr")], color='r', alpha=0.1, ls=':');
    ax[0,1].plot(f[experiment].timebounds, f[experiment].cumulative_emissions.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}")], color='k', alpha=0.1);
    ax[0,1].plot(f[experiment].cumulative_emissions.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}_zec")], color='b', alpha=0.1);
    # # ax[0,1].plot(f[cal].cumulative_emissions.loc[dict(specie="CO2", scenario="esm-flat40_cdr")], color='r', alpha=0.1, ls=':');
    ax[1,0].plot(f[experiment].timebounds, f[experiment].concentration.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}")], color='k', alpha=0.1);
    ax[1,0].plot(f[experiment].concentration.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}_zec")], color='b', alpha=0.1);
    # # ax[1,0].plot(f[cal].concentration.loc[dict(specie="CO2", scenario="esm-flat40_cdr")], color='r', alpha=0.1, ls=':');
    ax[1,1].plot(f[experiment].timebounds, f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}")], color='k', alpha=0.1);
    ax[1,1].plot(f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec")], color='b', alpha=0.1);
    # # ax[1,1].plot(f[cal].temperature.loc[dict(layer=0, scenario="esm-flat40_cdr")], color='r', alpha=0.1, ls=':');

In [ ]:
for experiment in emissions_rate:
    fig, ax = pl.subplots()
    pl.plot(f[experiment].timebounds, f[experiment].airborne_fraction.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}")], color='k', alpha=0.1);
    pl.plot(f[experiment].timebounds, f[experiment].airborne_fraction.loc[dict(specie="CO2", scenario=f"esm-flat{experiment}_zec")], color='b', alpha=0.1);
    # pl.plot(np.arange(0, 75), f[cal].airborne_fraction.loc[dict(specie="CO2", scenario="esm-flat40_cdr", timebounds=np.arange(0, 75))], color='r', alpha=0.1, ls=':');

In [ ]:
tcre = {}
zec50 = {}
zec100 = {}
zec200 = {}
total50 = {}
# tr1000 = {}
# tr0 = {}
# tpw = {}

for experiment in emissions_rate:
    ramp_up_length = int(1000/experiment)
    # TCRE is just warming at year 25
    tcre[experiment] = f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}", timebounds=ramp_up_length)]

    # ZEC50 is just warming at year 75 minus year 25
    zec50[experiment] = (
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length+50)] - 
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length)]
    )

    # ZEC100 is just warming at year 125 minus year 25
    zec100[experiment] = (
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length+100)] - 
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length)]
    )

    # ZEC200 is just warming at year 225 minus year 25
    zec200[experiment] = (
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length+200)] - 
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length)]
    )

    # total50 is the total warming 50 years after emissions reach zero (defined as TCRE + ZEC50)
    total50[experiment] = (
        f[experiment].temperature.loc[dict(layer=0, scenario=f"esm-flat{experiment}_zec", timebounds=ramp_up_length+50)]
    )
    
    # # TNZ can be calculated as a 20 year average around year 150 in esm-flat10-cdr minus a 20 year average around year 125 in esm-flat10
    # tnz = (
    #     f.temperature.loc[dict(layer=0, scenario="esm-flat10_cdr", timebounds=150)] - 
    #     f.temperature.loc[dict(layer=0, scenario="esm-flat10", timebounds=125)]
    # )

    # # TR1000 can be calculated as a 20 year average around year 200 in esm-flat10-cdr minus a 20 year average around year 100 in esm-flat10
    # tr1000[cal] = (
    #     f[cal].temperature.loc[dict(layer=0, scenario="esm-flat10_cdr", timebounds=200)] - 
    #     f[cal].temperature.loc[dict(layer=0, scenario="esm-flat10", timebounds=100)]
    # )

    # # TR0 can be calculated as a 20 year average around year 310 in esm-flat10-cdr
    # tr0[cal] = f[cal].temperature.loc[dict(layer=0, scenario="esm-flat10_cdr", timebounds=310)]

    # # Time to Peak Warming (tPW) can be calculated as the time difference between the peak value of 20-year smoothed global mean 
    # # temperatures and the point that net zero is achieved in esm-flat10-cdr (year 150)
    # tpw[cal] = f[cal].temperature.loc[dict(layer=0, scenario="esm-flat10_cdr")].argmax(axis=0) - 150

In [ ]:
#zec50

In [ ]:
df = {}
for experiment in emissions_rate:
    df[experiment] = pd.DataFrame(
        {
            "tcre": tcre[experiment],
            "zec50": zec50[experiment],
            "zec100": zec100[experiment],
            "zec200": zec200[experiment],
            "total50": total50[experiment],
            # "tr1000": tr1000[cal],
            # "tr0": tr0[cal],
            # "tpw": tpw[cal],
        },
        index = f[experiment].configs
    )

In [ ]:
df[40]["total50"].quantile((.05, .50, .95))

In [ ]:
df[20]["total50"].quantile((.05, .50, .95))

In [ ]:
df[10]["total50"].quantile((.05, .50, .95))

In [ ]:
df[5]["total50"].quantile((.05, .50, .95))

In [ ]:
df[2]["total50"].quantile((.05, .50, .95))

In [ ]:
df[40]["zec50"].quantile((.05, .50, .95))

In [ ]:
df[20]["zec50"].quantile((.05, .50, .95))

In [ ]:
df[10]["zec50"].quantile((.05, .50, .95))

In [ ]:
df[5]["zec50"].quantile((.05, .50, .95))

In [ ]:
df[2]["zec50"].quantile((.05, .50, .95))

In [ ]:
df[40]["zec100"].quantile((.05, .50, .95))

In [ ]:
df[20]["zec100"].quantile((.05, .50, .95))

In [ ]:
df[10]["zec100"].quantile((.05, .50, .95))

In [ ]:
df[5]["zec100"].quantile((.05, .50, .95))

In [ ]:
df[2]["zec100"].quantile((.05, .50, .95))

In [ ]:
df[40]["tcre"].quantile((.05, .50, .95))

In [ ]:
df[20]["tcre"].quantile((.05, .50, .95))

In [ ]:
df[10]["tcre"].quantile((.05, .50, .95))

In [ ]:
df[5]["tcre"].quantile((.05, .50, .95))

In [ ]:
df[2]["tcre"].quantile((.05, .50, .95))

In [ ]:
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["tcre"].quantile((.05)), 
        df[5]["tcre"].quantile((.05)), 
        df[10]["tcre"].quantile((.05)), 
        df[20]["tcre"].quantile((.05)), 
        df[40]["tcre"].quantile((.05))
    ],
    ls = '--',
    marker='o',
    color='r',
),
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["tcre"].quantile((.95)), 
        df[5]["tcre"].quantile((.95)), 
        df[10]["tcre"].quantile((.95)), 
        df[20]["tcre"].quantile((.95)), 
        df[40]["tcre"].quantile((.95))
    ],
    ls = '--',
    marker='o',
    color='r',
)
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["tcre"].quantile((.5)), 
        df[5]["tcre"].quantile((.5)), 
        df[10]["tcre"].quantile((.5)), 
        df[20]["tcre"].quantile((.5)), 
        df[40]["tcre"].quantile((.5))
    ],
    ls = '-',
    marker='o',
    color='k',
)
pl.xlabel('Emissions rate, GtC/yr')
pl.ylabel('TCRE, K at 1000 GtC')
pl.savefig('../plots/TCRE_flatX.png')

In [ ]:
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["total50"].quantile((.05)), 
        df[5]["total50"].quantile((.05)), 
        df[10]["total50"].quantile((.05)), 
        df[20]["total50"].quantile((.05)), 
        df[40]["total50"].quantile((.05))
    ],
    ls = '--',
    marker='o',
    color='r',
),
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["total50"].quantile((.95)), 
        df[5]["total50"].quantile((.95)), 
        df[10]["total50"].quantile((.95)), 
        df[20]["total50"].quantile((.95)), 
        df[40]["total50"].quantile((.95))
    ],
    ls = '--',
    marker='o',
    color='r',
)
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["total50"].quantile((.5)), 
        df[5]["total50"].quantile((.5)), 
        df[10]["total50"].quantile((.5)), 
        df[20]["total50"].quantile((.5)), 
        df[40]["total50"].quantile((.5))
    ],
    ls = '-',
    marker='o',
    color='k',
)
pl.xlabel('Emissions rate, GtC/yr')
pl.ylabel('TCRE + ZEC50, K at 1000 GtC')
pl.savefig('../plots/TCRE+ZEC50_flatX.png')

In [ ]:
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec50"].quantile((.05)), 
        df[5]["zec50"].quantile((.05)), 
        df[10]["zec50"].quantile((.05)), 
        df[20]["zec50"].quantile((.05)), 
        df[40]["zec50"].quantile((.05))
    ],
    ls = '--',
    marker='o',
    color='r',
),
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec50"].quantile((.95)), 
        df[5]["zec50"].quantile((.95)), 
        df[10]["zec50"].quantile((.95)), 
        df[20]["zec50"].quantile((.95)), 
        df[40]["zec50"].quantile((.95))
    ],
    ls = '--',
    marker='o',
    color='r',
)
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec50"].quantile((.5)), 
        df[5]["zec50"].quantile((.5)), 
        df[10]["zec50"].quantile((.5)), 
        df[20]["zec50"].quantile((.5)), 
        df[40]["zec50"].quantile((.5))
    ],
    ls = '-',
    marker='o',
    color='k',
)
pl.xlabel('Emissions rate, GtC/yr')
pl.ylabel('ZEC50, K after 1000 GtC')
pl.savefig('../plots/ZEC50_flatX.png')

In [ ]:
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec100"].quantile((.05)), 
        df[5]["zec100"].quantile((.05)), 
        df[10]["zec100"].quantile((.05)), 
        df[20]["zec100"].quantile((.05)), 
        df[40]["zec100"].quantile((.05))
    ],
    ls = '--',
    marker='o',
    color='r',
),
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec100"].quantile((.95)), 
        df[5]["zec100"].quantile((.95)), 
        df[10]["zec100"].quantile((.95)), 
        df[20]["zec100"].quantile((.95)), 
        df[40]["zec100"].quantile((.95))
    ],
    ls = '--',
    marker='o',
    color='r',
)
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec100"].quantile((.5)), 
        df[5]["zec100"].quantile((.5)), 
        df[10]["zec100"].quantile((.5)), 
        df[20]["zec100"].quantile((.5)), 
        df[40]["zec100"].quantile((.5))
    ],
    ls = '-',
    marker='o',
    color='k',
)
pl.xlabel('Emissions rate, GtC/yr')
pl.ylabel('zec100, K after 1000 GtC')
pl.savefig('../plots/ZEC100_flatX.png')

In [ ]:
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec200"].quantile((.05)), 
        df[5]["zec200"].quantile((.05)), 
        df[10]["zec200"].quantile((.05)), 
        df[20]["zec200"].quantile((.05)), 
        df[40]["zec200"].quantile((.05))
    ],
    ls = '--',
    marker='o',
    color='r',
),
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec200"].quantile((.95)), 
        df[5]["zec200"].quantile((.95)), 
        df[10]["zec200"].quantile((.95)), 
        df[20]["zec200"].quantile((.95)), 
        df[40]["zec200"].quantile((.95))
    ],
    ls = '--',
    marker='o',
    color='r',
)
pl.plot(
    np.array([2.5, 5, 10, 20, 40]),
    [
        df[2]["zec200"].quantile((.5)), 
        df[5]["zec200"].quantile((.5)), 
        df[10]["zec200"].quantile((.5)), 
        df[20]["zec200"].quantile((.5)), 
        df[40]["zec200"].quantile((.5))
    ],
    ls = '-',
    marker='o',
    color='k',
)
pl.xlabel('Emissions rate, GtC/yr')
pl.ylabel('zec200, K after 1000 GtC')
pl.savefig('../plots/ZEC200_flatX.png')

In [ ]:
for experiment in emissions_rate:
    sns.pairplot(
        df[experiment],
        corner=True,
        plot_kws={"alpha": 0.5},
        height=1,
    )
    pl.suptitle(f'{experiment} GtC/yr')

In [ ]:
# os.makedirs('../output/', exist_ok=True)
# for cal in calibrations:
#     df[cal].to_csv(f'../output/flat40_key-metrics_fair2.1.3_cal{cal}.csv')

In [ ]:
# for cal in calibrations:
#     ds = xr.Dataset(
#         data_vars=dict(
#             temperature=(["time", "scenario", "config"], f[cal].temperature.loc[dict(layer=0)].data),
#             co2_concentration=(["time", "scenario", "config"], f[cal].concentration.loc[dict(specie="CO2")].data),
#             airborne_fraction=(["time", "scenario", "config"], f[cal].airborne_fraction.loc[dict(specie="CO2")].data),
#             ecs=(["config"], f[cal].ebms.ecs.data),
#             tcr=(["config"], f[cal].ebms.tcr.data),
#             tcre=(["config"], tcre[cal].data),
#             zec50=(["config"], zec50[cal].data),
#             zec100=(["config"], zec100[cal].data),
#             zec200=(["config"], zec200[cal].data),
#             # tr1000=(["config"], tr1000[cal].data),
#             # tr0=(["config"], tr0[cal].data),
#             # tpw=(["config"], tpw[cal].data),
#         ),
#         coords=dict(
#             time=np.arange(226),
#             config=list(cal_df[cal].index),
#             scenario=scenarios
#         ),
#     )
#     ds.to_netcdf(f'../output/flat40_all-output_fair2.1.3_cal{cal}.nc')